In [1]:
from bs4 import BeautifulSoup
import httpx

In [2]:
from llama_cpp.llama import Llama, LlamaGrammar
import httpx
from llama_index.core.node_parser import HTMLNodeParser

from llama_cpp.llama import LlamaGrammar
import numpy as np
import pandas as pd
import torch
# from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.llms.llama_cpp.llama_utils import (
    messages_to_prompt,
    completion_to_prompt,
)
from llama_index.core import Settings
from llama_index.core import SimpleDirectoryReader, StorageContext
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.postgres import PGVectorStore
import textwrap
from llama_index.core import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from transformers import AutoTokenizer
from llama_index.core import set_global_tokenizer

from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import RouterQueryEngine

/llm/.venv/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_url" in LlamaCPP has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/llm/.venv/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_path" in LlamaCPP has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/llm/.venv/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_kwargs" in LlamaCPP has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/llm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.read

In [3]:
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct")

llm = LlamaCPP(
    # You can pass in the URL to a GGML model to download it automatically
    # optionally, you can set the path to a pre-downloaded model instead of model_url
    # model_path="/hf_cache/models--NousResearch--Hermes-3-Llama-3.1-8B-GGUF/snapshots/307a5dfb59aa38d88b6cfd32f44b8ad7c1da9fb8/Hermes-3-Llama-3.1-8B.Q5_K_M.gguf",
    # model_url="https://huggingface.co/bartowski/DeepSeek-Coder-V2-Lite-Instruct-GGUF/blob/main/DeepSeek-Coder-V2-Lite-Instruct-Q6_K.gguf",
    model_path="/hf_cache/models--bartowski--DeepSeek-Coder-V2-Lite-Instruct-GGUF/snapshots/8f248fa2072348f77a8bc37754e470de1f61866e/DeepSeek-Coder-V2-Lite-Instruct-Q6_K.gguf",
    temperature=0,
    max_new_tokens=4096,
    context_window=int(16384),
    generate_kwargs={
        # "repeat_penalty": 1.1,
        "top_k": 0,
        "top_p": 0
    },
    model_kwargs={
        "n_gpu_layers": -1,
        # "grammar": grammar
                 },
    # messages_to_prompt=messages_to_prompt,
    # completion_to_prompt=completion_to_prompt,
    verbose=False,
)

Settings.llm = llm

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

In [4]:
vector_store = PGVectorStore.from_params(
    database='grover',
    host='postgres',
    password='grover',
    port=5432,
    user='grover',
    table_name="crow_html",
    embed_dim=384,  
    hnsw_kwargs={
        "hnsw_m": 16,
        "hnsw_ef_construction": 64,
        "hnsw_ef_search": 40,
        "hnsw_dist_method": "vector_cosine_ops",
    },
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [5]:
html = SimpleDirectoryReader("/notebooks/llm/crow_rag/crowdocs/html/").load_data()


In [6]:
r = httpx.get("https://monome.org/docs/crow/reference/")

In [7]:
soup = BeautifulSoup(r.text)

In [8]:
tags = [tag.name for tag in soup.find_all()]

In [9]:
# By default, it will parse a select subset of HTML tags, but you can override this.

# The default tags are: ["p", "h1", "h2", "h3", "h4", "h5", "h6", "li", "b", "i", "u", "section"]


# # tags = all_classes
parser = HTMLNodeParser(tags=tags)
nodes = parser.get_nodes_from_documents(html)


In [10]:
len(nodes)

892

In [11]:
index = VectorStoreIndex(nodes, storage_context=storage_context, show_progress=True)

Generating embeddings: 100%|██████████| 892/892 [00:27<00:00, 32.25it/s] 


In [12]:
print(index.as_query_engine(response_mode="tree_summarize").query("What is ASL"))


ASL stands for "Abstract State Language". It is a type of language used in programming and computer science to describe the state of a system or a program. In the provided context, ASL is described as not being a program itself, meaning that variables within an ASL are fixed when the ASL is created. However, these variables can be updated by scripts or the REPL (Read-Eval-Print Loop) and will be used by the running ASL.
